# E-Commerce Review Prediction
**Author:** Hunter Sarkis  
**Dataset:** Olist Brazilian E-Commerce Public Dataset  
**Framework:** CRISP-DM

---

## Abstract
This project applies the CRISP-DM framework to analyze 100,000+ e-commerce orders from Olist, a Brazilian online retail marketplace. The business objective is to predict whether a customer will leave a negative review based on order characteristics such as price, freight cost, delivery time, and product category. By identifying the key drivers of dissatisfaction, retailers can proactively intervene before a bad review is submitted.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, chi2_contingency

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

Libraries loaded.


## Step 2: Load & Merge Data

We load the six relevant CSV files and merge them into one master dataframe.
Each order becomes one row with its price, freight, payment type, product category,
review score, and review comment.

In [2]:
# Load CSVs
orders    = pd.read_csv('olist_orders_dataset.csv')
items     = pd.read_csv('olist_order_items_dataset.csv')
reviews   = pd.read_csv('olist_order_reviews_dataset.csv')
payments  = pd.read_csv('olist_order_payments_dataset.csv')
products  = pd.read_csv('olist_products_dataset.csv')
cat_trans = pd.read_csv('product_category_name_translation.csv')

# Aggregate items per order (one row per order)
items_agg = items.groupby('order_id').agg(
    price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    product_id=('product_id', 'first')
).reset_index()

# Aggregate payments per order
payments_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', 'first')
).reset_index()

# Translate product categories to English
products_cat = products.merge(cat_trans, on='product_category_name', how='left')

# One review per order
reviews_dedup = reviews.drop_duplicates(subset='order_id', keep='first')

# Build master dataframe
df = (orders
      .merge(reviews_dedup[['order_id','review_score','review_comment_message']], on='order_id', how='inner')
      .merge(items_agg, on='order_id', how='left')
      .merge(payments_agg, on='order_id', how='left')
      .merge(products_cat[['product_id','product_category_name_english']], on='product_id', how='left')
     )

df.rename(columns={'product_category_name_english': 'product_category'}, inplace=True)

print('Master dataframe shape:', df.shape)
df.head()

Master dataframe shape: (98673, 16)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_comment_message,price,freight_value,product_id,payment_value,payment_type,product_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,4,"Não testei o produto ainda, mas ele veio corre...",29.99,8.72,87285b34884572647811a353c7ac498a,38.71,credit_card,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,4,Muito bom o produto.,118.70,22.76,595fac2a385ac33a80bd5114aec74eb8,141.46,boleto,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,5,NaN,159.90,19.22,aa4383b373c6aca5d8797843e5594415,179.12,credit_card,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,5,O produto foi exatamente o que eu esperava e e...,45.00,27.20,d0b61bfb1de832b15ba9d266ca96e5b0,72.20,credit_card,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,5,NaN,19.90,8.72,65266b2da20d04dbe00c5c2d3bb7859e,28.62,credit_card,stationery
